# ② Hán OCR (PaddleOCR + Qwen-VL consensus) — Minh Mệnh Chính Yếu  ·  notebook P2 / P3

Reads NB1's `output_p1/<VOL>.zip`, re-OCRs the vertical Hán with PaddleOCR, then cleans it with a
Qwen2.5-VL consensus arbiter (step 3b). Hands a corrected `ocr_zips/<VOL>.zip` per volume to
**notebook 3** (bge-m3 alignment). No Surya, no bge here — those stacks conflict over Pillow.

**Run order:** NB1 (VI OCR) → **this** → NB3 (alignment). Needs `output_p1/vol1..6.zip` from NB1.


## 1. Dependencies (paddle + Qwen-VL — NO surya, NO bge)  →  then **Runtime ▸ Restart**

In [ ]:
# Hán OCR (paddle) + consensus (Qwen2.5-VL). Surya AND bge intentionally absent (bge lives in notebook 3).
!pip -q install PyMuPDF opencv-python-headless pandas openpyxl tqdm huggingface_hub
# consensus arbiter: Qwen2.5-VL + helpers (opencc folds traditional/simplified for voting)
!pip -q install "transformers>=4.49" qwen-vl-utils accelerate bitsandbytes opencc-python-reimplemented
!pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!pip install -U "paddleocr[doc-parser]>=3.4.0"
# REPAIR torch NCCL — MUST be last (paddle-gpu downgrades it, breaks `import torch`).
!pip install -U nvidia-nccl-cu12
print('\n>>> Now: Runtime ▸ Restart session, then run VERIFY.')


### Verify paddle + torch GPU (after restart)

In [ ]:
import numpy, paddle, torch
print('numpy', numpy.__version__, '| paddle', paddle.__version__, '| torch', torch.__version__)
assert paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, \
    'paddle GPU not visible — Runtime ▸ GPU, reinstall, restart'
assert torch.cuda.is_available(), 'torch CUDA broken — re-run nvidia-nccl-cu12, restart'
paddle.set_device('gpu')
print('paddle GPU conv OK:', tuple(paddle.nn.Conv2D(3,8,3)(paddle.randn([1,3,32,32])).shape))
print('torch CUDA OK:', torch.zeros(1).cuda().device)

## 2. Mount Drive + project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/minh_menh'
MMCY  = '/content/mmcy'                       # LOCAL work root (same as notebooks 1 & 3)
os.environ['MMCY_ROOT'] = MMCY
%cd $DRIVE
!mkdir -p {MMCY}/out
!rsync -a {DRIVE}/assets/ {MMCY}/assets/      # dicts (local, for build_dicts + step3)
print('code:', DRIVE, '| work root (local):', MMCY)


## 3. Build dictionaries (one-time; downloads Unihan ~8 MB — needed for âm Hán-Việt)

In [ ]:
!python -m pipeline.build_dicts


## 4. Run ALL 6 volumes — OCR + consensus, per-vol push to Drive

In [ ]:
# ── Setup for the 6-vol loop (Hán OCR only — no bge here) ──
from pathlib import Path
import numpy as np
from pipeline import config
from pipeline.common import read_jsonl, write_jsonl

VOLS = ["vol1", "vol2", "vol3", "vol4", "vol5", "vol6"]   # edit to run a subset
print("will OCR:", VOLS)


In [ ]:
# ── Consensus setup: Qwen2.5-VL arbiter, load ONCE, reuse every vol ──
# Cleans Hán OCR before hand-off: Qwen-VL re-reads each column image and overrules base
# PaddleOCR where they disagree. Two engines only — base (step-3 PaddleOCR) + Qwen arbiter.
# (The PP-OCRv5 "spec" specialist was dropped: ~0.4% agreement, never decides the vote,
# and costs a model download + a forward pass per column.) Pure vote / re-transliterate
# logic is in pipeline/han_consensus.py; this cell only loads Qwen and crops columns.
# Toggle USE_CONSENSUS=False to hand off raw PaddleOCR. Knobs live in config.CONSENSUS.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # fight VRAM fragmentation
USE_CONSENSUS = True
import re, time
from PIL import Image
from pipeline import han_consensus as hc
from pipeline.step3_sinonom import load_hanviet

CN = config.CONSENSUS
HANVIET = load_hanviet()
_HAN = re.compile(r"[\u3400-\u9fff\uf900-\ufaff]")

if USE_CONSENSUS:
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info
    _vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
                if torch.cuda.is_available() else 0)
    _use_4bit = CN["load_4bit"] and _vram_gb < CN["fp16_vram_gb"]
    _qkw = dict(torch_dtype=torch.float16, device_map="auto")
    if _use_4bit:
        _qkw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    print(f"GPU {_vram_gb:.0f} GB -> Qwen {'4-bit' if _use_4bit else 'fp16'}")
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(CN["qwen_model"], **_qkw).eval()
    # cap vision tokens so a big merged/雙行 crop can't blow up attention memory
    qproc  = AutoProcessor.from_pretrained(CN["qwen_model"],
                                           max_pixels=CN["qwen_max_pixels"],
                                           min_pixels=CN["qwen_min_pixels"])
    _PROMPT = ("图中是古籍木刻版的一列繁体汉字，单列竖排。请严格按从上到下的顺序逐字识别，"
               "不得颠倒次序、不得增字或漏字。只输出繁体汉字本身，不要任何解释、标点、拼音、数字或多余文字。")

    _ROT = {"none": None, "cw": Image.ROTATE_270, "ccw": Image.ROTATE_90}
    # LRU-bounded page cache: columns are processed in page order, so only the
    # last few pages are ever reused. Unbounded caching held an ENTIRE volume of
    # decoded RGB pages (~12 MB each x hundreds = GB) -> host RAM climbed all run.
    from collections import OrderedDict
    _PCACHE_MAX = 3
    _pcache = OrderedDict()
    def _page(pdir, pg):
        key = (pdir, pg)
        im = _pcache.get(key)
        if im is None:
            im = Image.open(f"{pdir}/page_{pg:04d}.png").convert("RGB")
            _pcache[key] = im
            if len(_pcache) > _PCACHE_MAX:
                _pcache.popitem(last=False)   # evict oldest
        else:
            _pcache.move_to_end(key)
        return im
    def _crop(pdir, b, rot="none"):
        im = _page(pdir, b["page"]); x0, y0, x1, y1 = b["bbox"]; k = CN["crop_inset"]
        c = im.crop((x0 + k, y0 + k, max(x0 + k + 1, x1 - k), max(y0 + k + 1, y1 - k)))
        if CN["upscale"] != 1:
            c = c.resize((c.width * CN["upscale"], c.height * CN["upscale"]), Image.LANCZOS)
        if _ROT.get(rot) is not None:
            c = c.transpose(_ROT[rot])
        return c
    @torch.no_grad()
    def _qwen_read(pdir, b):
        pil = _crop(pdir, b); n = len(b.get("sinonom", "") or "")
        prompt = _PROMPT + (f"这一列大约有 {n} 个字。" if n else "")
        msgs = [{"role": "user", "content": [{"type": "image", "image": pil},
                                             {"type": "text", "text": prompt}]}]
        text = qproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, vids = process_vision_info(msgs)
        inp = qproc(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(qmodel.device)
        out = qmodel.generate(**inp, max_new_tokens=CN["max_new_tok"], do_sample=False)
        gen = out[:, inp.input_ids.shape[1]:]
        return "".join(_HAN.findall(qproc.batch_decode(gen, skip_special_tokens=True)[0]))

    def run_consensus(boxes, pdir):
        """Qwen re-reads the cols worth re-reading (all in vlm_mode='full', else the
        low-confidence ones); base==qwen keeps base, a disagreement trusts qwen.
        Returns (consensus_records, review_records, {id: corrected_text})."""
        _pcache.clear()                                   # free previous vol's page images
        if CN["vlm_mode"] == "full":
            need = boxes
        else:                                             # gate the VLM on base confidence
            g = CN["qwen_conf_gate"]
            need = [b for b in boxes if (b.get("conf") is None or b["conf"] < g)]
        if CN.get("qwen_skip_dbl", True):                 # 雙行 order is a guess -> don't re-read
            need = [b for b in need if not b.get("is_dbl")]
        ec = CN.get("qwen_empty_cache_every", 0)
        print(f"   Qwen will read {len(need)}/{len(boxes)} cols (vlm_mode={CN['vlm_mode']}, skip_dbl={CN.get('qwen_skip_dbl', True)})")
        t0 = time.time()
        for n, b in enumerate(need):
            b["_qwen"] = _qwen_read(pdir, b)
            if ec and (n + 1) % ec == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()                  # fight fragmentation over the long loop
            if (n + 1) % 100 == 0:
                el = time.time() - t0
                print(f"   qwen {n+1}/{len(need)} ({el:.0f}s, ETA {el/(n+1)*(len(need)-n-1)/60:.0f}m)")
        for b in boxes:
            b.setdefault("_qwen", "")
        recs = [{"id": b["id"], "page": b["page"], "conf": b.get("conf"),
                 "base": b.get("sinonom", ""), "qwen": b["_qwen"]} for b in boxes]
        cons, review = hc.build_consensus(recs, vote_mode=CN["vote_mode"])
        corrected = {r["id"]: r["consensus"] for r in cons}
        _pcache.clear()                                   # drop this vol's page images now
        import gc as _gc; _gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return cons, review, corrected

    print("consensus ready:", CN["qwen_model"], "| base + qwen only | vlm_mode:", CN["vlm_mode"])
else:
    def run_consensus(boxes, pdir):
        return [], [], {}
    print("USE_CONSENSUS = False -> handing off raw PaddleOCR")


In [ ]:
# ── Step 3c setup: Hán auto-punctuation (sentence segmentation), load ONCE, reuse ──
# Woodblock text is unpunctuated -> step3 falls back to column-as-sentence (~3x over-seg
# vs VI). After consensus fixes chars, a token-classification punctuator re-reads each
# page's clean main-column stream; we split on sentence-final marks into REAL sentences.
# Pure split + box-mapping is in pipeline.han_segment; this cell only loads the model
# and defines han_labels(text)->[per-char mark]. 雙行 cols are left unpunctuated (tagged).
from pipeline import config, han_segment as hseg
from pipeline.common import make_id

HS = config.HAN_SEGMENT
HS_ENABLED = HS["enabled"]
if HS_ENABLED:
    import torch
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    _ptok = AutoTokenizer.from_pretrained(HS["model"])
    _pmodel = AutoModelForTokenClassification.from_pretrained(HS["model"]).eval()
    if torch.cuda.is_available():
        _pmodel = _pmodel.to("cuda")
    _ID2LAB = _pmodel.config.id2label

    @torch.no_grad()
    def _labels_chunk(text):
        """Per-char predicted mark for one <=512 chunk. '' when label is O/none."""
        enc = _ptok(text, return_tensors="pt", return_offsets_mapping=True,
                    truncation=True, max_length=512)
        off = enc.pop("offset_mapping")[0].tolist()
        logits = _pmodel(**{k: v.to(_pmodel.device) for k, v in enc.items()}).logits[0]
        pred = logits.argmax(-1).tolist()
        marks = [""] * len(text)
        for (s, e), pi in zip(off, pred):
            if e <= s:                                   # special tokens ([CLS]/[SEP])
                continue
            lab = _ID2LAB[pi]
            if lab and lab != "O":
                marks[e - 1] = lab                       # mark follows the char ending at e-1
        return marks

    def han_labels(text):
        """Per-char marks for a full page stream; chunk with overlap if > max_len."""
        n = len(text); mx = HS["max_len"]; ov = HS["overlap"]
        if n <= mx:
            return _labels_chunk(text)
        marks = [""] * n; i = 0
        while i < n:
            j = min(i + mx, n)
            cm = _labels_chunk(text[i:j])
            # trust the non-overlap tail of each chunk (overlap head is context only)
            lo = i if i == 0 else i + ov
            for k in range(lo, j):
                marks[k] = cm[k - i]
            if j >= n:
                break
            i = j - ov
        return marks
    print("punctuator ready:", HS["model"])
else:
    han_labels = None
    print("HAN_SEGMENT disabled -> keep column-as-sentence")


In [ ]:
# ── OCR every volume; push EACH vol's corrected zip to Drive the moment it finishes ──
# Reads NB1 hand-off output_p1/<VOL>.zip, runs step3 (PaddleOCR) + step3b (Qwen-VL consensus),
# then the Hán review queue (char-validation on the CORRECTED chars — moved out of P3),
# re-zips the WHOLE working folder (VI passthrough + corrected Hán) to ocr_zips/<VOL>.zip for notebook 3.
import traceback, time, json as _json
!mkdir -p {DRIVE}/ocr_zips
done, failed = [], []
for idx, VOL in enumerate(VOLS):
    print(f"\n{'='*64}\n#### {VOL}  ####\n{'='*64}")
    t0 = time.time()
    try:
        # ── hand-off from NB1: ONE zip -> unzip onto LOCAL ssd (Drive FUSE corrupts) ──
        IN_DIR = config.OUT_DIR / VOL
        !cp -f {DRIVE}/output_p1/{VOL}.zip {MMCY}/out/
        !cd {MMCY}/out && unzip -qo {VOL}.zip
        assert (IN_DIR / "pages_han").exists(), \
            f"{IN_DIR}/pages_han missing — run NB1 first (writes output_p1/{VOL}.zip)"

        # ── step 3 — Hán OCR (base = PaddleOCR) ──
        !python -m pipeline.step3_sinonom --vol {VOL}
        assert (IN_DIR / "han_sentences.jsonl").exists(), \
            f"step3 produced no han_sentences.jsonl for {VOL}"

        han_boxes = list(read_jsonl(IN_DIR / "han_boxes.jsonl"))
        han_sents = list(read_jsonl(IN_DIR / "han_sentences.jsonl"))

        # ── step 3b — consensus clean: Qwen-VL arbiter fixes base BEFORE hand-off ──
        if USE_CONSENSUS and han_boxes:
            cons, review, corrected = run_consensus(han_boxes, str(IN_DIR / "pages_han"))
            qrun = sum(1 for b in han_boxes if b.get("_qwen"))          # count before stripping
            han_boxes = hc.apply_consensus(han_boxes, corrected, HANVIET)   # for char-val + Excel
            write_jsonl(IN_DIR / "han_boxes.jsonl", han_boxes)
            # step 3c — re-segment into REAL sentences from the CORRECTED boxes
            if HS_ENABLED:
                han_sents = hseg.resegment_boxes(han_boxes, han_labels, hc.transliterate,
                                                 HANVIET, make_id, HS["sent_marks"])
            else:
                han_sents = hc.apply_consensus(han_sents, corrected, HANVIET)
            write_jsonl(IN_DIR / "han_sentences.jsonl", han_sents)
            print(f"segment: {len(han_boxes)} cols -> {len(han_sents)} sentences "
                  f"(dbl {sum(1 for x in han_sents if x.get('is_dbl'))})")
            write_jsonl(IN_DIR / "consensus.jsonl", cons)
            write_jsonl(IN_DIR / "review_queue.jsonl", review)
            mtr = hc.consensus_metrics(cons, review, VOL, CN["vote_mode"], CN["vlm_mode"], qrun)
            (IN_DIR / "consensus_metrics.json").write_text(
                _json.dumps(mtr, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"consensus: {mtr['cols']} cols | corrected {mtr['corrected_chars']} chars "
                  f"| review {mtr['review_cols']} cols | auto-settle {mtr['auto_settle_char_pct']}%")

        # ── Hán review queue — char-validation (S1∩S2) + low-conf boxes on the
        #    CORRECTED chars. Moved out of P3 so P3 is alignment-only. Writes
        #    han_review.jsonl (rides along in the zip).
        !python -m pipeline.step3_sinonom --vol {VOL} --review

        # ── push THIS vol's corrected folder to Drive now (FUSE-safe single zip) ──
        # carries VI passthrough (pages_vi, vi_*.jsonl, vi_review, split_manifest) + corrected
        # Hán + han_review + consensus
        !cd {MMCY}/out && zip -qr {DRIVE}/ocr_zips/{VOL}.zip {VOL}
        !ls -lh {DRIVE}/ocr_zips/{VOL}.zip
        print(f"✅ {VOL} OCR DONE in {(time.time()-t0)/60:.1f} min -> Drive ocr_zips/{VOL}.zip")
        done.append(VOL)
    except Exception as e:
        traceback.print_exc()
        print(f"❌ {VOL} FAILED: {e} — continuing to next vol")
        failed.append(VOL)

print(f"\n{'='*64}\nSUMMARY  done={done}  failed={failed}")
